# 3W Toolkit v3 — Dados reais não rotulados → pré-processamento → Windowing → entrada para MOMENT

O objetivo é testar **somente a preparação dos dados**. Não haverá treinamento, avaliação ou uso da classe `Pipeline`.

Fluxo:

```text
Parquets reais não rotulados
        ↓
ParquetDatasetConfig
        ↓
CleanSignals
        ↓
ImputeMissing
        ↓
Normalize
        ↓
Windowing
        ↓
arrays/sequências para o futuro fine-tuning do MOMENT
```

### Decisões importantes

- Os arquivos são tratados como **eventos reais**.
- Não usamos labels.
- Não usamos `SequentialPreprocessingAdapterConfig`.
- Não usamos `Pipeline`.
- Cada etapa de pré-processamento do Toolkit é executada separadamente.
- O `WindowingConfig` do Toolkit é usado diretamente através de `TransformConfig`.
- O janelamento é feito **depois** do pré-processamento.
- As janelas nunca atravessam o limite entre dois arquivos Parquet.
- O notebook mantém metadados para saber de qual arquivo cada janela veio.

No Overview, o Toolkit mostra `ParquetDatasetConfig(...).build()` para carregar os dados, `CleanSignalsConfig`, `ImputeMissingConfig`, `NormalizeConfig` para pré-processamento e `WindowingConfig` para dividir uma série em segmentos sobrepostos. O exemplo de Windowing usa `TransformConfig(..., feature_extraction=WindowingConfig(window_size=128))`. 


## 1. Configuração

Aponte `DATA_PATH` para a pasta com os Parquets reais.

O Toolkit reconhece eventos reais pelo nome dos arquivos (`WELL...`) na implementação original, mas como nossa base externa pode não seguir exatamente essa convenção, este notebook usa a versão de carregamento de Parquets não rotulados criada anteriormente e depois converte cada instância para o formato esperado pelas transformações do Toolkit. 

Se os nomes dos arquivos forem compatíveis com o padrão 3W, também é possível usar `event_type=["real"]` diretamente no `ParquetDatasetConfig` do Toolkit. O Overview demonstra exatamente essa filtragem.

In [ ]:
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, Sequence

import numpy as np
import pandas as pd

from ThreeWToolkit.dataset import ParquetDatasetConfig, TransformConfig

from ThreeWToolkit.preprocessing import (
    ImputeMissingConfig,
    NormalizeConfig,
    CleanSignalsConfig,
)

from ThreeWToolkit.feature_extraction import WindowingConfig


# ============================================================
# DADOS
# ============================================================

DATA_PATH = Path("/home/bruno.martins/dataset")

# None = usar todas as colunas numéricas do primeiro arquivo.
# Exemplo:
# SIGNAL_COLUMNS = ["P-PDG", "P-TPT", "T-TPT"]
SIGNAL_COLUMNS: Optional[list[str]] = None

RECURSIVE = True

# Quantidade de arquivos usada nos testes iniciais.
N_FILES_TO_TEST = 5


# ============================================================
# PRÉ-PROCESSAMENTO
# ============================================================

# As etapas são executadas separadamente.
# USE_CLEAN_SIGNALS = True
IMPUTE_STRATEGY = "mean"
NORMALIZE_NORM = "l2"


# ============================================================
# JANELAMENTO
# ============================================================

WINDOW_SIZE = 128

# Percentual de sobreposição entre janelas.
# 0.0 = sem sobreposição
# 0.5 = 50% de sobreposição
OVERLAP = 0.0

PAD_LAST_WINDOW = False
PAD_VALUE = 0.0


# ============================================================
# MOMENT
# ============================================================

# O objetivo aqui é preparar as sequências.
# Não fazemos nenhuma chamada ao modelo MOMENT neste notebook.
EXPORT_PATH = Path("./moment_input.npz")


## 2. Carregamento dos Parquets

O Toolkit representa cada evento como um objeto com `signal`, `label` e `metadata`. A implementação original de `load_file()` lê um único Parquet e retorna exatamente essa estrutura.

Para os nossos dados externos, usamos uma classe pequena equivalente, sem exigir labels.

In [2]:
@dataclass
class DatasetOutputs:
    signal: pd.DataFrame
    label: object = None
    metadata: dict = field(default_factory=dict)


@dataclass
class UnlabeledParquetDatasetConfig:
    path: Path | str
    columns: Optional[Sequence[str]] = None
    recursive: bool = True

    def __post_init__(self):
        self.path = Path(self.path)


class UnlabeledParquetDataset:
    """Carrega Parquets reais sem exigir labels."""

    def __init__(self, config: UnlabeledParquetDatasetConfig):
        self.config = config
        self.root = Path(config.path)

        if not self.root.exists():
            raise FileNotFoundError(
                f"Pasta não encontrada: {self.root.resolve()}"
            )

        pattern = "**/*.parquet" if config.recursive else "*.parquet"
        self.files_events = sorted(self.root.glob(pattern))

        if not self.files_events:
            raise FileNotFoundError(
                f"Nenhum .parquet encontrado em {self.root.resolve()}"
            )

    def __len__(self):
        return len(self.files_events)

    def __getitem__(self, idx):
        return self.load_file(idx)

    def load_file(self, idx):
        path = self.files_events[idx]

        df = pd.read_parquet(path, engine="pyarrow")

        if self.config.columns is not None:
            columns = list(self.config.columns)
            missing = [c for c in columns if c not in df.columns]

            if missing:
                raise ValueError(
                    f"{path.name}: colunas ausentes: {missing}"
                )

            df = df.loc[:, columns].copy()
        else:
            df = df.copy()

        return DatasetOutputs(
            signal=df,
            label=None,
            metadata={
                "file_name": path.name,
                "relative_path": str(path.relative_to(self.root)),
                "event_type": "real",
                "event_class": None,
                "labeled": False,
            },
        )


dataset_config = UnlabeledParquetDatasetConfig(
    path=DATA_PATH,
    columns=SIGNAL_COLUMNS,
    recursive=RECURSIVE,
)

raw_dataset = UnlabeledParquetDataset(dataset_config)

print(f"Dataset: {raw_dataset.root.resolve()}")
print(f"Arquivos: {len(raw_dataset)}")


Dataset: /nfs/home/bruno.martins/dataset
Arquivos: 58


## 3. Seleção das variáveis

In [3]:
first_event = raw_dataset[0]

if SIGNAL_COLUMNS is None:
    signal_columns = first_event.signal.select_dtypes(
        include=[np.number]
    ).columns.tolist()
else:
    signal_columns = list(SIGNAL_COLUMNS)

if not signal_columns:
    raise ValueError(
        "Nenhuma coluna numérica foi encontrada."
    )

print("Variáveis selecionadas:")
for column in signal_columns:
    print(" -", column)

print(f"\nTotal: {len(signal_columns)} variáveis")

# Verificar se todos os arquivos possuem essas colunas.
schema_errors = []

for idx in range(len(raw_dataset)):
    event = raw_dataset[idx]
    missing = [
        c for c in signal_columns
        if c not in event.signal.columns
    ]

    if missing:
        schema_errors.append({
            "file": event.metadata["relative_path"],
            "missing": missing,
        })

if schema_errors:
    display(pd.DataFrame(schema_errors))
    raise ValueError(
        "Existem arquivos sem todas as variáveis selecionadas."
    )

print("[OK] Todas as instâncias possuem as variáveis selecionadas.")


Variáveis selecionadas:
 - ABER-CKGL
 - ABER-CKP
 - ESTADO-DHSV
 - ESTADO-M1
 - ESTADO-M2
 - ESTADO-PXO
 - ESTADO-SDV-GL
 - ESTADO-SDV-P
 - ESTADO-W1
 - ESTADO-W2
 - ESTADO-XO
 - P-ANULAR
 - P-JUS-BS
 - P-JUS-CKGL
 - P-JUS-CKP
 - P-MON-CKGL
 - P-MON-CKP
 - P-MON-SDV-P
 - P-PDG
 - PT-P
 - P-TPT
 - QBS
 - QGL
 - T-JUS-CKP
 - T-MON-CKP
 - T-PDG
 - T-TPT

Total: 27 variáveis
[OK] Todas as instâncias possuem as variáveis selecionadas.


## 4. Adaptador para o formato do Toolkit

As transformações do Overview são aplicadas sobre datasets que retornam objetos com `.signal`. Por isso, mantemos exatamente essa interface para que `TransformConfig` possa operar sobre nossos arquivos externos.

In [4]:
class ToolkitDatasetView:
    """View do dataset externo com somente as variáveis escolhidas."""

    def __init__(self, dataset, columns):
        self.dataset = dataset
        self.columns = list(columns)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        event = self.dataset[idx]

        return DatasetOutputs(
            signal=event.signal.loc[:, self.columns].copy(),
            label=None,
            metadata=dict(event.metadata),
        )

    def __iter__(self):
        for idx in range(len(self)):
            yield self[idx]


toolkit_dataset = ToolkitDatasetView(
    raw_dataset,
    signal_columns,
)

print(
    f"Dataset compatível com Toolkit: "
    f"{len(toolkit_dataset)} eventos"
)


Dataset compatível com Toolkit: 58 eventos


## 5. Inspeção antes do pré-processamento

Primeiro verificamos o estado original de um evento.

In [5]:
event_before = toolkit_dataset[0]

print("Arquivo:", event_before.metadata["relative_path"])
print("Shape:", event_before.signal.shape)
print("NaNs:", int(event_before.signal.isna().sum().sum()))

display(event_before.signal.head())

assert isinstance(event_before.signal, pd.DataFrame)
assert event_before.label is None

print("[OK] Estrutura inicial correta.")


Arquivo: WELL-00005_20170331050014.parquet
Shape: (235318, 27)
NaNs: 2589203


,ABER-CKGL,ABER-CKP,ESTADO-DHSV,ESTADO-M1,ESTADO-M2,ESTADO-PXO,ESTADO-SDV-GL,ESTADO-SDV-P,ESTADO-W1,ESTADO-W2,...,P-MON-SDV-P,P-PDG,PT-P,P-TPT,QBS,QGL,T-JUS-CKP,T-MON-CKP,T-PDG,T-TPT
timestamp,,,,,,,,,,,,,,,,,,,,,
2017-03-31 05:00:14,0.0,28.371908,0.0,1.0,1.0,0.0,NaN,1.0,1.0,0.0,...,NaN,NaN,NaN,2.079164e+07,NaN,NaN,67.509010,NaN,NaN,106.372238
2017-03-31 05:01:14,0.0,28.372513,0.0,1.0,1.0,0.0,NaN,1.0,1.0,0.0,...,NaN,NaN,NaN,2.079373e+07,NaN,NaN,67.507919,NaN,NaN,106.369843
2017-03-31 05:02:14,0.0,28.373117,0.0,1.0,1.0,0.0,NaN,1.0,1.0,0.0,...,NaN,NaN,NaN,2.079185e+07,NaN,NaN,67.506828,NaN,NaN,106.368790
2017-03-31 05:03:14,0.0,28.373722,0.0,1.0,1.0,0.0,NaN,1.0,1.0,0.0,...,NaN,NaN,NaN,2.079457e+07,NaN,NaN,67.505737,NaN,NaN,106.369843
2017-03-31 05:04:14,0.0,28.374325,0.0,1.0,1.0,0.0,NaN,1.0,1.0,0.0,...,NaN,NaN,NaN,2.079499e+07,NaN,NaN,67.504646,NaN,NaN,106.365341


[OK] Estrutura inicial correta.


# 6. Pré-processamento com o Toolkit

Todo o pré-processamento será aplicado de uma vez utilizando
`SequentialPreprocessingAdapterConfig`.

A sequência será:

1. `CleanSignalsConfig()`
2. `ImputeMissingConfig(strategy="mean")`
3. `NormalizeConfig(norm="l2")`

Depois do `fit()`, o dataset transformado será utilizado como entrada
para o janelamento.

In [9]:
from ThreeWToolkit.preprocessing import (
    CleanSignalsConfig,
    ImputeMissingConfig,
    NormalizeConfig,
    SequentialPreprocessingAdapterConfig,
) 

pipeline = SequentialPreprocessingAdapterConfig(
    steps=[
        CleanSignalsConfig(),
        ImputeMissingConfig(strategy=IMPUTE_STRATEGY),
        NormalizeConfig(norm=NORMALIZE_NORM)
    ]
)

transformer = TransformConfig(pre_processing=pipeline).build()
transformer.fit(toolkit_dataset)
transformed_ds = transformer.transform(toolkit_dataset)

print("Pipeline applied successfully.")

Pipeline applied successfully.


## 7. Validar o pré-processamento completo

Agora percorremos os arquivos e verificamos se:

- o número de variáveis foi preservado;
- o número de amostras foi preservado;
- os dados são numéricos;
- não existem `NaN`/`Inf` depois do pré-processamento.

In [10]:
preprocessing_results = []

for idx in range(len(transformed_ds)):
    event = transformed_ds[idx]
    values = event.signal.to_numpy(dtype=np.float32)

    result = {
        "file": event.metadata["relative_path"],
        "rows": values.shape[0],
        "variables": values.shape[1],
        "nan": int(np.isnan(values).sum()),
        "inf": int(np.isinf(values).sum()),
        "finite": bool(np.isfinite(values).all()),
    }

    preprocessing_results.append(result)

preprocessing_results = pd.DataFrame(
    preprocessing_results
)

display(preprocessing_results.head(20))

assert (preprocessing_results["nan"] == 0).all()
assert (preprocessing_results["inf"] == 0).all()
assert preprocessing_results["finite"].all()

print(
    f"[OK] {len(preprocessing_results)} "
    "arquivos passaram pelo pré-processamento."
)


,file,rows,variables,nan,inf,finite
0,WELL-00005_20170331050014.parquet,235318,17,0,0,True
1,WELL-00007_20170517200012.parquet,109560,17,0,0,True
2,WELL-00008_20170610210246.parquet,137994,17,0,0,True
3,WELL-00009_20170313150804.parquet,173,17,0,0,True
4,WELL-00011_20140515083000.parquet,207886,17,0,0,True
5,WELL-00012_20170320011000.parquet,858,17,0,0,True
6,WELL-00013_20170329010229.parquet,231,17,0,0,True
7,WELL-00022_20180802233838.parquet,132742,17,0,0,True
8,WELL-00023_20180826212652.parquet,96393,17,0,0,True
9,WELL-00024_20160704180000.parquet,166450,17,0,0,True


[OK] 58 arquivos passaram pelo pré-processamento.


# 8. Windowing usando o Toolkit

Agora usamos a **`WindowingConfig` do próprio Toolkit**, em vez de implementar o janelamento manualmente.

O Overview descreve:

- `window_size`: número de amostras;
- `overlap`: razão de sobreposição;
- `pad_last_window`: se a última janela incompleta deve ser preenchida;
- `pad_value`: valor do preenchimento.

O exemplo oficial usa:

```python
TransformConfig(
    pre_processing=...,
    feature_extraction=WindowingConfig(window_size=128),
).build()
```

e depois `fit()`/`transform()`.

Aqui não incluímos nenhum pipeline de treinamento e não usamos `SequentialFeatureAdapterConfig`: somente a transformação de Windowing.

In [11]:
windowing_transformer = TransformConfig(
    feature_extraction=WindowingConfig(
        window_size=WINDOW_SIZE,
        overlap=OVERLAP,
        pad_last_window=PAD_LAST_WINDOW,
        pad_value=PAD_VALUE,
    )
).build()

# Windowing não precisa aprender parâmetros do treinamento,
# mas seguimos a mesma interface mostrada no Overview.
windowing_transformer.fit(transformed_ds)

windowed_dataset = windowing_transformer.transform(
    transformed_ds
)

print("[OK] Windowing executado.")


[OK] Windowing executado.


## 9. Inspecionar o resultado do Windowing

O Overview observa que, depois do Windowing, cada linha corresponde a um segmento windowed da série original. fileciteturn1file0L129-L132

Aqui precisamos descobrir exatamente como a versão instalada do Toolkit representa essas janelas (`DataFrame`, arrays dentro das células etc.). Por isso a célula abaixo inspeciona o resultado sem assumir antecipadamente um formato.

In [19]:
windowed_event = windowed_dataset[0]

print("Arquivo:", windowed_event.metadata["relative_path"])
print("Tipo de signal:", type(windowed_event.signal))
print("Shape:", windowed_event.signal.shape)

print("\nColunas:")
print(windowed_event.signal.columns)

print("\nÍndice:")
print(windowed_event.signal.index)

print("\nNíveis do índice:")
print(windowed_event.signal.index.names)

print("\nPrimeira janela:")
first_window = windowed_event.signal.iloc[0]

print("Tipo:", type(first_window))
print("Shape:", first_window.shape)
print(first_window)

Arquivo: WELL-00005_20170331050014.parquet
Tipo de signal: <class 'pandas.core.frame.DataFrame'>
Shape: (31263, 128)

Colunas:
RangeIndex(start=0, stop=128, step=1)

Índice:
MultiIndex([(   0,   'ESTADO-DHSV'),
            (   0,     'ESTADO-M1'),
            (   0,     'ESTADO-M2'),
            (   0,    'ESTADO-PXO'),
            (   0, 'ESTADO-SDV-GL'),
            (   0,  'ESTADO-SDV-P'),
            (   0,     'ESTADO-W1'),
            (   0,     'ESTADO-W2'),
            (   0,     'ESTADO-XO'),
            (   0,      'P-ANULAR'),
            ...
            (1838,     'ESTADO-W2'),
            (1838,     'ESTADO-XO'),
            (1838,      'P-ANULAR'),
            (1838,    'P-JUS-CKGL'),
            (1838,     'P-MON-CKP'),
            (1838,         'P-PDG'),
            (1838,         'P-TPT'),
            (1838,           'QGL'),
            (1838,     'T-JUS-CKP'),
            (1838,         'T-TPT')],
           names=['window', 'variable'], length=31263)

Níveis do índ

In [20]:
windowed_signal = windowed_event.signal

window_variables = (
    windowed_signal.index
    .get_level_values(1)
    .unique()
    .tolist()
)

n_variables = len(window_variables)
window_size = windowed_signal.shape[1]
n_rows = len(windowed_signal)

assert n_rows % n_variables == 0

n_windows = n_rows // n_variables

print("Variáveis após pré-processamento:")
print(window_variables)

print("\nNúmero de variáveis:", n_variables)
print("Window size:", window_size)
print("Linhas no DataFrame:", n_rows)
print("Número de janelas:", n_windows)

Variáveis após pré-processamento:
['ESTADO-DHSV', 'ESTADO-M1', 'ESTADO-M2', 'ESTADO-PXO', 'ESTADO-SDV-GL', 'ESTADO-SDV-P', 'ESTADO-W1', 'ESTADO-W2', 'ESTADO-XO', 'P-ANULAR', 'P-JUS-CKGL', 'P-MON-CKP', 'P-PDG', 'P-TPT', 'QGL', 'T-JUS-CKP', 'T-TPT']

Número de variáveis: 17
Window size: 128
Linhas no DataFrame: 31263
Número de janelas: 1839


## 10. Converter o resultado do Toolkit para o formato de entrada

O objetivo final deste notebook não é treinar o MOMENT, mas obter uma estrutura limpa:

```text
X.shape = (n_janelas, window_size, n_variaveis)
```

Como a representação exata pode variar conforme a versão do Toolkit, a função abaixo trata os formatos mais comuns produzidos pelo `Windowing`.

Ela **não cria janelas manualmente**: somente reorganiza o resultado produzido pelo Toolkit.

In [22]:
X_first = windowed_signal.to_numpy(
    dtype=np.float32
).reshape(
    n_windows,
    n_variables,
    window_size,
)

print("Shape antes do transpose:", X_first.shape)

Shape antes do transpose: (1839, 17, 128)


In [23]:
X_first = np.transpose(
    X_first,
    (0, 2, 1),
)

print("Shape final:", X_first.shape)

assert X_first.shape == (
    n_windows,
    window_size,
    n_variables,
)

assert np.isfinite(X_first).all()

print("[OK] Conversão realizada com sucesso.")

Shape final: (1839, 128, 17)
[OK] Conversão realizada com sucesso.


## 11. Testar a conversão da primeira instância

In [24]:
first_window = X_first[0]

print("Shape da primeira janela:", first_window.shape)

first_window_df = pd.DataFrame(
    first_window,
    columns=windowed_signal.index
        .get_level_values("variable")
        .unique()
)

display(first_window_df.head(10))

Shape da primeira janela: (128, 17)


variable,ESTADO-DHSV,ESTADO-M1,ESTADO-M2,ESTADO-PXO,ESTADO-SDV-GL,ESTADO-SDV-P,ESTADO-W1,ESTADO-W2,ESTADO-XO,P-ANULAR,P-JUS-CKGL,P-MON-CKP,P-PDG,P-TPT,QGL,T-JUS-CKP,T-TPT
0,0.0,1.0,1.0,0.0,0.356601,1.0,1.0,0.0,0.0,1.909384,0.912553,0.0,0.0,2.267745,3.156601e-16,0.73687,0.753821
1,0.0,1.0,1.0,0.0,0.356601,1.0,1.0,0.0,0.0,1.909384,0.912553,0.0,0.0,2.267745,3.156601e-16,0.73687,0.753821
2,0.0,1.0,1.0,0.0,0.356601,1.0,1.0,0.0,0.0,1.909384,0.912553,0.0,0.0,2.267745,3.156601e-16,0.73687,0.753821
3,0.0,1.0,1.0,0.0,0.356601,1.0,1.0,0.0,0.0,1.909384,0.912553,0.0,0.0,2.267745,3.156601e-16,0.73687,0.753821
4,0.0,1.0,1.0,0.0,0.356601,1.0,1.0,0.0,0.0,1.909384,0.912553,0.0,0.0,2.267745,3.156601e-16,0.73687,0.753821
5,0.0,1.0,1.0,0.0,0.356601,1.0,1.0,0.0,0.0,1.909384,0.912553,0.0,0.0,2.267745,3.156601e-16,0.73687,0.753821
6,0.0,1.0,1.0,0.0,0.356601,1.0,1.0,0.0,0.0,1.909384,0.912553,0.0,0.0,2.267745,3.156601e-16,0.73687,0.753821
7,0.0,1.0,1.0,0.0,0.356601,1.0,1.0,0.0,0.0,1.909384,0.912553,0.0,0.0,2.267745,3.156601e-16,0.73687,0.753821
8,0.0,1.0,1.0,0.0,0.356601,1.0,1.0,0.0,0.0,1.909384,0.912553,0.0,0.0,2.267745,3.156601e-16,0.73687,0.753821
9,0.0,1.0,1.0,0.0,0.356601,1.0,1.0,0.0,0.0,1.909384,0.912553,0.0,0.0,2.267745,3.156601e-16,0.73687,0.753821


In [25]:
assert first_window_df.shape == (
    WINDOW_SIZE,
    n_variables,
)

print("[OK] Primeira janela possui o formato esperado.")

[OK] Primeira janela possui o formato esperado.


In [26]:
# Seleciona diretamente do DataFrame do Toolkit
# a primeira janela da primeira variável.

first_variable = (
    windowed_signal
    .index
    .get_level_values("variable")
    .unique()[0]
)

toolkit_first_series = windowed_signal.loc[
    (0, first_variable)
].to_numpy(dtype=np.float32)

# Recupera a mesma série do nosso X
converted_first_series = X_first[
    0,
    :,
    0
]

print("Variável:", first_variable)
print("Shape Toolkit:", toolkit_first_series.shape)
print("Shape convertido:", converted_first_series.shape)

assert np.array_equal(
    toolkit_first_series,
    converted_first_series,
)

print("[OK] Os valores foram preservados durante a conversão.")

Variável: ESTADO-DHSV
Shape Toolkit: (128,)
Shape convertido: (128,)
[OK] Os valores foram preservados durante a conversão.


=============================================================
Continuar daqui
=============================================================

## 12. Gerar as janelas de todos os arquivos

Cada arquivo é processado separadamente.

Isso é importante porque **não queremos que uma janela contenha o final de um poço/arquivo e o início de outro**.

In [ ]:
all_windows = []
window_metadata = []
failed_files = []

for file_idx in range(len(windowed_dataset)):
    event = windowed_dataset[file_idx]

    try:
        X_event = windowed_signal_to_numpy(
            event.signal,
            window_size=WINDOW_SIZE,
            n_variables=len(signal_columns),
        )

        assert X_event.ndim == 3
        assert X_event.shape[1] == WINDOW_SIZE
        assert X_event.shape[2] == len(signal_columns)
        assert np.isfinite(X_event).all()

        for window_idx in range(X_event.shape[0]):
            all_windows.append(X_event[window_idx])

            window_metadata.append({
                "file_idx": file_idx,
                "file_name": event.metadata["file_name"],
                "relative_path": event.metadata["relative_path"],
                "window_idx": window_idx,
            })

    except Exception as exc:
        failed_files.append({
            "file_idx": file_idx,
            "file": event.metadata["relative_path"],
            "error": f"{type(exc).__name__}: {exc}",
        })


if failed_files:
    print("Arquivos que falharam:")
    display(pd.DataFrame(failed_files))

    raise RuntimeError(
        f"{len(failed_files)} arquivo(s) não puderam "
        "ser convertidos para janelas."
    )

if all_windows:
    X = np.stack(all_windows).astype(np.float32)
else:
    X = np.empty(
        (0, WINDOW_SIZE, len(signal_columns)),
        dtype=np.float32,
    )

window_metadata = pd.DataFrame(window_metadata)

print("==========================================")
print("RESULTADO DO JANELAMENTO")
print("==========================================")
print("X.shape:", X.shape)
print("X.dtype:", X.dtype)
print("Metadata:", window_metadata.shape)

assert X.ndim == 3
assert X.shape[1] == WINDOW_SIZE
assert X.shape[2] == len(signal_columns)
assert X.shape[0] == len(window_metadata)
assert np.isfinite(X).all()

print("\n[OK] Todos os arquivos foram convertidos.")


## 13. Inspeção de uma janela

Aqui visualizamos uma janela individual como `DataFrame` para confirmar que:

- as variáveis estão na ordem esperada;
- a janela possui exatamente `WINDOW_SIZE` amostras;
- os valores estão finitos.

In [ ]:
if len(X) > 0:
    first_window_df = pd.DataFrame(
        X[0],
        columns=signal_columns,
    )

    print("Shape:", first_window_df.shape)
    display(first_window_df.head(10))

    print("\nEstatísticas:")
    display(first_window_df.describe())

    assert first_window_df.shape == (
        WINDOW_SIZE,
        len(signal_columns),
    )

    print("[OK] Janela individual válida.")
else:
    print(
        "Nenhuma janela foi gerada. "
        "Verifique se os eventos possuem pelo menos "
        f"{WINDOW_SIZE} amostras."
    )


## 14. Verificar quantas janelas cada arquivo gerou

Essa tabela ajuda a identificar arquivos curtos demais ou diferenças de comprimento entre eventos.

In [ ]:
windows_per_file = (
    window_metadata
    .groupby(
        ["file_idx", "file_name", "relative_path"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "n_windows"})
)

display(windows_per_file.head(30))

print(
    f"Arquivos com pelo menos uma janela: "
    f"{len(windows_per_file)}"
)
print(
    f"Total de janelas: {len(X)}"
)


## 15. Salvar uma amostra no formato que será usado futuramente

Não treinamos o MOMENT aqui.

Apenas salvamos:

- `X`: janelas;
- `file_name`: arquivo de origem;
- `window_idx`: índice da janela;
- `signal_columns`: nomes e ordem das variáveis.

O `.npz` é apenas um formato de teste/intercâmbio. O formato definitivo para o fine-tuning pode ser escolhido depois.

In [ ]:
if len(X) > 0:
    np.savez_compressed(
        EXPORT_PATH,
        X=X,
        file_name=window_metadata["file_name"].to_numpy(),
        relative_path=window_metadata["relative_path"].to_numpy(),
        window_idx=window_metadata["window_idx"].to_numpy(),
        signal_columns=np.asarray(signal_columns),
    )

    print(
        f"[OK] Arquivo salvo em: "
        f"{EXPORT_PATH.resolve()}"
    )

    # Teste de leitura.
    loaded = np.load(
        EXPORT_PATH,
        allow_pickle=True,
    )

    X_loaded = loaded["X"]

    assert X_loaded.shape == X.shape
    assert np.allclose(X_loaded, X)

    print("[OK] Arquivo salvo e recarregado corretamente.")
else:
    print("Nada foi salvo porque nenhuma janela foi gerada.")


# 16. Validação final

O teste final verifica o contrato que queremos entregar ao próximo estágio:

```text
X
├── dimensão 0 = janelas
├── dimensão 1 = WINDOW_SIZE
└── dimensão 2 = variáveis/sensores
```

Nenhum treinamento ou pipeline do Toolkit é executado.

In [ ]:
print("==============================================")
print(" VALIDAÇÃO FINAL")
print("==============================================")

assert len(raw_dataset) > 0
assert len(signal_columns) > 0
assert len(transformed_ds) == len(raw_dataset)
assert len(windowed_dataset) == len(raw_dataset)

assert X.ndim == 3
assert X.shape[1] == WINDOW_SIZE
assert X.shape[2] == len(signal_columns)

assert len(window_metadata) == X.shape[0]
assert np.isfinite(X).all()

print(f"Arquivos carregados:       {len(raw_dataset)}")
print(f"Variáveis:                 {len(signal_columns)}")
print(f"Window size:               {WINDOW_SIZE}")
print(f"Overlap:                   {OVERLAP}")
print(f"Total de janelas:          {X.shape[0]}")
print(f"Shape final:               {X.shape}")
print(f"NaN final:                 {np.isnan(X).sum()}")
print(f"Inf final:                 {np.isinf(X).sum()}")

print("\nPré-processamento:")
print(" - CleanSignals:", USE_CLEAN_SIGNALS)
print(" - ImputeMissing:", IMPUTE_STRATEGY)
print(" - Normalize:", NORMALIZE_NORM)

print("\n==============================================")
print(" PIPELINE DE DADOS: PASSOU")
print("==============================================")
print("Parquet → Toolkit preprocessing → Toolkit Windowing → X")
print("Nenhum treinamento foi executado.")


## 17. Resultado esperado para o MOMENT

Ao final, o principal objeto é:

```python
X
```

com:

```python
X.shape
# (n_windows, WINDOW_SIZE, n_variables)
```

Esse é o **produto deste notebook**: uma coleção de sequências windowed, pré-processadas e rastreáveis ao arquivo original.

### Observação importante

O notebook deliberadamente não assume ainda o formato final exigido pelo código de fine-tuning do MOMENT. Primeiro queremos comprovar que o **3W Toolkit consegue carregar, limpar, imputar, normalizar e janelar os dados reais**.

```text
X do Toolkit
   ↓
adaptação para o formato do MOMENT
   ↓
Dataset/DataLoader
   ↓
fine-tuning
```